# 1.Load Prompts

In [1]:
import pickle
import numpy as np
import random
from sklearn.neighbors import NearestNeighbors
from utils import convert_arrays_to_shapes

In [3]:
# load the pre-built router dataset 
with open(f'C:\\Users\\zqy\\OneDrive\\leaderboard_new_prompt.pkl', 'rb') as f:
    prompt_new = pickle.load(f)

with open(f'C:\\Users\\zqy\\OneDrive\\leaderboard_old_prompt.pkl', 'rb') as f:
    prompt_old = pickle.load(f)
    
with open(f'C:\\Users\\zqy\\OneDrive\\leaderboard_mmlu_pro_prompt.pkl', 'rb') as f:
    prompt_mmlu_pro = pickle.load(f)

with open(f'C:\\Users\\zqy\\OneDrive\\gsm8k_router_dataset.pkl', 'rb') as f:
    router_dataset4 = pickle.load(f)

print(prompt_new.keys())
print(prompt_old.keys())
print(prompt_mmlu_pro.keys())

# display the pre-built router dataset 
print(convert_arrays_to_shapes(prompt_new))
#print(router_dataset['gpqa_extended'])
print(convert_arrays_to_shapes(prompt_old))
#print(router_dataset2['harness_winogrande_5'])
print(convert_arrays_to_shapes(prompt_mmlu_pro))
print(convert_arrays_to_shapes(router_dataset4))

dict_keys(['gpqa_extended', 'bbh_formal_fallacies', 'ifeval', 'bbh_tracking_shuffled_objects_five_objects', 'math_algebra_hard', 'musr_murder_mysteries', 'bbh_disambiguation_qa', 'bbh_hyperbaton', 'gpqa_main', 'bbh_geometric_shapes', 'bbh_object_counting', 'bbh_web_of_lies', 'bbh_tracking_shuffled_objects_three_objects', 'math_intermediate_algebra_hard', 'bbh_movie_recommendation', 'musr_team_allocation', 'bbh_logical_deduction_seven_objects', 'math_prealgebra_hard', 'bbh_boolean_expressions', 'gpqa_diamond', 'bbh_ruin_names', 'bbh_penguins_in_a_table', 'bbh_snarks', 'bbh_logical_deduction_five_objects', 'bbh_logical_deduction_three_objects', 'bbh_navigate', 'math_geometry_hard', 'bbh_reasoning_about_colored_objects', 'math_num_theory_hard', 'bbh_causal_judgement', 'musr_object_placements', 'bbh_tracking_shuffled_objects_seven_objects', 'math_precalculus_hard', 'bbh_sports_understanding', 'math_counting_and_prob_hard', 'bbh_date_understanding', 'bbh_temporal_sequences', 'bbh_salient_tr

# 2.Build Prompt Records

In [18]:
prompt_records = []

# -------- 第一部分：旧数据 --------
old_dataset_to_group = {
    'harness_winogrande_5': 'winogrande',
    'harness_arc_challenge_25': 'arc',
    'harness_gsm8k_5': 'gsm8k',
    'harness_truthfulqa_mc_0': 'truthfulqa',
    'harness_hellaswag_10': 'hellaswag',
}

old_mmlu_datasets = [
    'harness_hendrycksTest_high_school_biology_5',
    'harness_hendrycksTest_philosophy_5',
    'harness_hendrycksTest_professional_accounting_5',
    'harness_hendrycksTest_human_aging_5',
    'harness_hendrycksTest_clinical_knowledge_5',
    'harness_hendrycksTest_college_medicine_5',
    'harness_hendrycksTest_virology_5',
    'harness_hendrycksTest_anatomy_5',
    'harness_hendrycksTest_public_relations_5',
    'harness_hendrycksTest_human_sexuality_5',
    'harness_hendrycksTest_security_studies_5',
    'harness_hendrycksTest_elementary_mathematics_5',
    'harness_hendrycksTest_medical_genetics_5',
    'harness_hendrycksTest_high_school_chemistry_5',
    'harness_hendrycksTest_prehistory_5',
    'harness_hendrycksTest_us_foreign_policy_5',
    'harness_hendrycksTest_high_school_microeconomics_5',
    'harness_hendrycksTest_high_school_government_and_politics_5',
    'harness_hendrycksTest_computer_security_5',
    'harness_hendrycksTest_global_facts_5',
    'harness_gsm8k_5',
    'harness_hendrycksTest_high_school_world_history_5',
    'harness_hendrycksTest_jurisprudence_5',
    'harness_hendrycksTest_astronomy_5',
    'harness_hendrycksTest_college_computer_science_5',
    'harness_hendrycksTest_high_school_computer_science_5',
    'harness_hendrycksTest_college_mathematics_5',
    'harness_hendrycksTest_sociology_5',
    'harness_hendrycksTest_machine_learning_5',
    'harness_hendrycksTest_conceptual_physics_5',
    'harness_hendrycksTest_high_school_mathematics_5',
    'harness_hendrycksTest_high_school_macroeconomics_5',
    'harness_hendrycksTest_high_school_us_history_5',
    'harness_hendrycksTest_professional_psychology_5',
    'harness_hendrycksTest_moral_disputes_5',
    'harness_hendrycksTest_management_5',
    'harness_hendrycksTest_international_law_5',
    'harness_hendrycksTest_high_school_physics_5',
    'harness_truthfulqa_mc_0',
    'harness_hendrycksTest_logical_fallacies_5',
    'harness_hendrycksTest_world_religions_5',
    'harness_hendrycksTest_professional_medicine_5',
    'harness_hendrycksTest_college_physics_5',
    'harness_hendrycksTest_formal_logic_5',
    'harness_hendrycksTest_nutrition_5',
    'harness_hendrycksTest_econometrics_5',
    'harness_hendrycksTest_high_school_geography_5',
    'harness_hendrycksTest_high_school_psychology_5',
    'harness_hendrycksTest_electrical_engineering_5',
    'harness_hellaswag_10',
    'harness_hendrycksTest_high_school_statistics_5',
    'harness_hendrycksTest_high_school_european_history_5',
    'harness_hendrycksTest_marketing_5',
    'harness_hendrycksTest_moral_scenarios_5',
    'harness_hendrycksTest_business_ethics_5',
    'harness_hendrycksTest_miscellaneous_5',
    'harness_hendrycksTest_professional_law_5',
    'harness_hendrycksTest_college_biology_5',
    'harness_hendrycksTest_abstract_algebra_5',
    'harness_hendrycksTest_college_chemistry_5',
]

for ds in old_mmlu_datasets:
    if ds not in old_dataset_to_group:
        old_dataset_to_group[ds] = 'mmlu'

for dataset_name, group_name in old_dataset_to_group.items():
    for prompt in prompt_old[dataset_name]:
        prompt_records.append({
            'prompt': prompt,
            'dataset_group': group_name,
            'dataset_name': dataset_name,
        })

# -------- 第二部分：新数据 --------
new_dataset_to_group = {
    'ifeval': 'ifeval',
    'gpqa_extended': 'gpqa',
    'gpqa_main': 'gpqa',
    'gpqa_diamond': 'gpqa',
    'musr_murder_mysteries': 'musr',
    'musr_team_allocation': 'musr',
    'musr_object_placements': 'musr',
    'math_algebra_hard': 'math_lvl5',
    'math_intermediate_algebra_hard': 'math_lvl5',
    'math_prealgebra_hard': 'math_lvl5',
    'math_geometry_hard': 'math_lvl5',
    'math_num_theory_hard': 'math_lvl5',
    'math_precalculus_hard': 'math_lvl5',
    'math_counting_and_prob_hard': 'math_lvl5',
    'bbh_formal_fallacies': 'bbh',
    'bbh_tracking_shuffled_objects_five_objects': 'bbh',
    'bbh_disambiguation_qa': 'bbh',
    'bbh_hyperbaton': 'bbh',
    'bbh_geometric_shapes': 'bbh',
    'bbh_object_counting': 'bbh',
    'bbh_web_of_lies': 'bbh',
    'bbh_tracking_shuffled_objects_three_objects': 'bbh',
    'bbh_movie_recommendation': 'bbh',
    'bbh_logical_deduction_seven_objects': 'bbh',
    'bbh_boolean_expressions': 'bbh',
    'bbh_ruin_names': 'bbh',
    'bbh_penguins_in_a_table': 'bbh',
    'bbh_snarks': 'bbh',
    'bbh_logical_deduction_five_objects': 'bbh',
    'bbh_logical_deduction_three_objects': 'bbh',
    'bbh_navigate': 'bbh',
    'bbh_reasoning_about_colored_objects': 'bbh',
    'bbh_causal_judgement': 'bbh',
    'musr_object_placements': 'musr',
    'bbh_tracking_shuffled_objects_seven_objects': 'bbh',
    'bbh_sports_understanding': 'bbh',
    'bbh_date_understanding': 'bbh',
    'bbh_temporal_sequences': 'bbh',
    'bbh_salient_translation_error_detection': 'bbh',
}

for dataset_name, group_name in new_dataset_to_group.items():
    for prompt in prompt_new[dataset_name]:
        prompt_records.append({
            'prompt': prompt,
            'dataset_group': group_name,
            'dataset_name': dataset_name,
        })

# -------- 第三部分：mmlu_pro --------
for prompt in prompt_mmlu_pro['mmlu_pro']:
    prompt_records.append({
        'prompt': prompt,
        'dataset_group': 'mmlu_pro',
        'dataset_name': 'mmlu_pro',
    })



prompt_list = [x['prompt'] for x in prompt_records]
prompt_dataset_list = [x['dataset_group'] for x in prompt_records]
prompt_dataset_name_list = [x['dataset_name'] for x in prompt_records]

In [19]:
import re
import json
import pandas as pd


# =========================================================
# 0. 基础清洗
# =========================================================
def clean_text(x):
    if x is None:
        return None
    x = str(x)
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = x.replace("\u00a0", " ")
    x = re.sub(r"[ \t]+\n", "\n", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    x = re.sub(r"[ \t]{2,}", " ", x)
    x = x.strip()
    return x if x else None


def json_dumps(obj):
    return json.dumps(obj, ensure_ascii=False)


# =========================================================
# 1. 构建 prompt_records
#    每个元素: {prompt, dataset_group, dataset_name}
# =========================================================
def build_prompt_records(prompt_old, prompt_new, prompt_mmlu_pro):
    prompt_records = []

    old_dataset_to_group = {
        "harness_winogrande_5": "winogrande",
        "harness_arc_challenge_25": "arc",
        "harness_gsm8k_5": "gsm8k",
        "harness_truthfulqa_mc_0": "truthfulqa",
        "harness_hellaswag_10": "hellaswag",
    }

    old_mmlu_datasets = [
        "harness_hendrycksTest_high_school_biology_5",
        "harness_hendrycksTest_philosophy_5",
        "harness_hendrycksTest_professional_accounting_5",
        "harness_hendrycksTest_human_aging_5",
        "harness_hendrycksTest_clinical_knowledge_5",
        "harness_hendrycksTest_college_medicine_5",
        "harness_hendrycksTest_virology_5",
        "harness_hendrycksTest_anatomy_5",
        "harness_hendrycksTest_public_relations_5",
        "harness_hendrycksTest_human_sexuality_5",
        "harness_hendrycksTest_security_studies_5",
        "harness_hendrycksTest_elementary_mathematics_5",
        "harness_hendrycksTest_medical_genetics_5",
        "harness_hendrycksTest_high_school_chemistry_5",
        "harness_hendrycksTest_prehistory_5",
        "harness_hendrycksTest_us_foreign_policy_5",
        "harness_hendrycksTest_high_school_microeconomics_5",
        "harness_hendrycksTest_high_school_government_and_politics_5",
        "harness_hendrycksTest_computer_security_5",
        "harness_hendrycksTest_global_facts_5",
        "harness_hendrycksTest_high_school_world_history_5",
        "harness_hendrycksTest_jurisprudence_5",
        "harness_hendrycksTest_astronomy_5",
        "harness_hendrycksTest_college_computer_science_5",
        "harness_hendrycksTest_high_school_computer_science_5",
        "harness_hendrycksTest_college_mathematics_5",
        "harness_hendrycksTest_sociology_5",
        "harness_hendrycksTest_machine_learning_5",
        "harness_hendrycksTest_conceptual_physics_5",
        "harness_hendrycksTest_high_school_mathematics_5",
        "harness_hendrycksTest_high_school_macroeconomics_5",
        "harness_hendrycksTest_high_school_us_history_5",
        "harness_hendrycksTest_professional_psychology_5",
        "harness_hendrycksTest_moral_disputes_5",
        "harness_hendrycksTest_management_5",
        "harness_hendrycksTest_international_law_5",
        "harness_hendrycksTest_high_school_physics_5",
        "harness_hendrycksTest_logical_fallacies_5",
        "harness_hendrycksTest_world_religions_5",
        "harness_hendrycksTest_professional_medicine_5",
        "harness_hendrycksTest_college_physics_5",
        "harness_hendrycksTest_formal_logic_5",
        "harness_hendrycksTest_nutrition_5",
        "harness_hendrycksTest_econometrics_5",
        "harness_hendrycksTest_high_school_geography_5",
        "harness_hendrycksTest_high_school_psychology_5",
        "harness_hendrycksTest_electrical_engineering_5",
        "harness_hendrycksTest_high_school_statistics_5",
        "harness_hendrycksTest_high_school_european_history_5",
        "harness_hendrycksTest_marketing_5",
        "harness_hendrycksTest_moral_scenarios_5",
        "harness_hendrycksTest_business_ethics_5",
        "harness_hendrycksTest_miscellaneous_5",
        "harness_hendrycksTest_professional_law_5",
        "harness_hendrycksTest_college_biology_5",
        "harness_hendrycksTest_abstract_algebra_5",
        "harness_hendrycksTest_college_chemistry_5",
    ]
    for ds in old_mmlu_datasets:
        old_dataset_to_group[ds] = "mmlu"

    for dataset_name, group_name in old_dataset_to_group.items():
        for prompt in prompt_old[dataset_name]:
            prompt_records.append({
                "prompt": prompt,
                "dataset_group": group_name,
                "dataset_name": dataset_name,
            })

    new_dataset_to_group = {
        "ifeval": "ifeval",

        "gpqa_extended": "gpqa",
        "gpqa_main": "gpqa",
        "gpqa_diamond": "gpqa",

        "musr_murder_mysteries": "musr",
        "musr_team_allocation": "musr",
        "musr_object_placements": "musr",

        "math_algebra_hard": "math_lvl5",
        "math_intermediate_algebra_hard": "math_lvl5",
        "math_prealgebra_hard": "math_lvl5",
        "math_geometry_hard": "math_lvl5",
        "math_num_theory_hard": "math_lvl5",
        "math_precalculus_hard": "math_lvl5",
        "math_counting_and_prob_hard": "math_lvl5",

        "bbh_formal_fallacies": "bbh",
        "bbh_tracking_shuffled_objects_five_objects": "bbh",
        "bbh_disambiguation_qa": "bbh",
        "bbh_hyperbaton": "bbh",
        "bbh_geometric_shapes": "bbh",
        "bbh_object_counting": "bbh",
        "bbh_web_of_lies": "bbh",
        "bbh_tracking_shuffled_objects_three_objects": "bbh",
        "bbh_movie_recommendation": "bbh",
        "bbh_logical_deduction_seven_objects": "bbh",
        "bbh_boolean_expressions": "bbh",
        "bbh_ruin_names": "bbh",
        "bbh_penguins_in_a_table": "bbh",
        "bbh_snarks": "bbh",
        "bbh_logical_deduction_five_objects": "bbh",
        "bbh_logical_deduction_three_objects": "bbh",
        "bbh_navigate": "bbh",
        "bbh_reasoning_about_colored_objects": "bbh",
        "bbh_causal_judgement": "bbh",
        "bbh_tracking_shuffled_objects_seven_objects": "bbh",
        "bbh_sports_understanding": "bbh",
        "bbh_date_understanding": "bbh",
        "bbh_temporal_sequences": "bbh",
        "bbh_salient_translation_error_detection": "bbh",
    }

    for dataset_name, group_name in new_dataset_to_group.items():
        for prompt in prompt_new[dataset_name]:
            prompt_records.append({
                "prompt": prompt,
                "dataset_group": group_name,
                "dataset_name": dataset_name,
            })

    for prompt in prompt_mmlu_pro["mmlu_pro"]:
        prompt_records.append({
            "prompt": prompt,
            "dataset_group": "mmlu_pro",
            "dataset_name": "mmlu_pro",
        })

    return prompt_records


# =========================================================
# 2. 头部清理
# =========================================================
MMLU_HEADER_RE = re.compile(
    r"^The following are multiple choice questions \(with answers\) about .+?\.\s*",
    flags=re.IGNORECASE | re.DOTALL,
)

GENERIC_INTRO_RE = re.compile(
    r"^(Here are some example questions from experts\.\s*"
    r"Answer the final question yourself, following the format of the previous questions exactly\.\s*)",
    flags=re.IGNORECASE | re.DOTALL,
)


def strip_known_global_header(text):
    if text is None:
        return None
    text = text.strip()
    text = re.sub(MMLU_HEADER_RE, "", text)
    text = re.sub(GENERIC_INTRO_RE, "", text)
    return text.strip()


# =========================================================
# 3. Answer 识别
# =========================================================
ANSWER_LINE_RE = re.compile(
    r"(?im)^(?:Answer|Final answer|答案)\s*[:：]\s*(.+?)\s*$"
)

HASH_ANSWER_LINE_RE = re.compile(
    r"(?im)^####\s*(.+?)\s*$"
)

THE_ANSWER_IS_RE = re.compile(
    r"(?i)\bThe answer is\b\s*:?\s*(.+?)(?:[.\n]|$)"
)

BOXED_RE = re.compile(
    r"\\boxed\{([^{}]+)\}"
)

LETTER_ONLY_RE = re.compile(r"^[A-Z]$", flags=re.IGNORECASE)
LETTER_PAREN_RE = re.compile(r"^\(?([A-Z])\)?$", flags=re.IGNORECASE)


def normalize_answer(ans):
    ans = clean_text(ans)
    if ans is None:
        return None

    ans = re.sub(r"^(?:Answer|Final answer|答案)\s*[:：]\s*", "", ans, flags=re.IGNORECASE)
    ans = re.sub(r"^####\s*", "", ans)
    ans = ans.strip()

    m = LETTER_PAREN_RE.match(ans)
    if m:
        return m.group(1).upper()
    return ans


def extract_explicit_answer(text):
    text = clean_text(text)
    if text is None:
        return None, None

    matches = list(ANSWER_LINE_RE.finditer(text))
    if matches:
        m = matches[-1]
        return normalize_answer(m.group(1)), (m.start(), m.end())

    matches = list(HASH_ANSWER_LINE_RE.finditer(text))
    if matches:
        m = matches[-1]
        return normalize_answer(m.group(1)), (m.start(), m.end())

    matches = list(THE_ANSWER_IS_RE.finditer(text))
    if matches:
        m = matches[-1]
        return normalize_answer(m.group(1)), (m.start(), m.end())

    return None, None


# =========================================================
# 4. 按 answer 行切多个 block
#    这是解决 MMLU few-shot 的关键
# =========================================================
def split_prompt_into_blocks(prompt, dataset_group):
    text = clean_text(prompt)
    if text is None:
        return []

    if dataset_group == "ifeval":
        return [text]

    text = strip_known_global_header(text)

    # 先找显式 answer line；每个 block 截止到当前 answer 行末尾
    answer_matches = list(ANSWER_LINE_RE.finditer(text)) + list(HASH_ANSWER_LINE_RE.finditer(text))
    answer_matches = sorted(answer_matches, key=lambda m: m.start())

    if answer_matches:
        blocks = []
        block_start = 0
        for m in answer_matches:
            block_end = m.end()
            block = text[block_start:block_end].strip()
            if block:
                blocks.append(block)
            block_start = block_end

        tail = text[block_start:].strip()
        if tail:
            blocks.append(tail)

        return [b for b in blocks if clean_text(b)]

    # 没有 answer 行，则不强拆
    return [text]


# =========================================================
# 5. question / options 拆分
# =========================================================
OPTION_LINE_RE = re.compile(
    r"^\s*(?:"
    r"(?P<label1>[A-Z])[\.\)]"      # A. / A)
    r"|"
    r"\((?P<label2>[A-Z])\)"        # (A)
    r"|"
    r"(?P<label3>\d+)[\.\)]"        # 1. / 1)
    r")\s*(?P<text>.+?)\s*$",
    flags=re.IGNORECASE,
)


def looks_like_option_label_sequence(labels):
    if len(labels) < 2:
        return False

    if all(label.isalpha() and len(label) == 1 for label in labels):
        ints = [ord(label.upper()) for label in labels]
        return ints == list(range(ints[0], ints[0] + len(ints)))

    if all(label.isdigit() for label in labels):
        ints = [int(x) for x in labels]
        return ints == list(range(ints[0], ints[0] + len(ints)))

    return False


def split_question_and_options(question_text):
    question_text = clean_text(question_text)
    if question_text is None:
        return None, []

    lines = question_text.split("\n")

    option_candidates = []
    for idx, line in enumerate(lines):
        m = OPTION_LINE_RE.match(line)
        if m:
            label = m.group("label1") or m.group("label2") or m.group("label3")
            option_candidates.append((idx, label.upper(), m.group("text").strip()))

    if len(option_candidates) < 2:
        return question_text, []

    labels = [x[1] for x in option_candidates]
    if not looks_like_option_label_sequence(labels):
        return question_text, []

    option_start_idx = option_candidates[0][0]
    stem = "\n".join(lines[:option_start_idx]).strip()

    options = []
    current = None
    for line in lines[option_start_idx:]:
        m = OPTION_LINE_RE.match(line)
        if m:
            if current is not None:
                current["text"] = clean_text(current["text"])
                options.append(current)
            label = m.group("label1") or m.group("label2") or m.group("label3")
            current = {
                "label": label.upper(),
                "text": m.group("text").strip(),
            }
        else:
            if current is not None:
                current["text"] += "\n" + line.strip()
            else:
                stem = (stem + "\n" + line.strip()).strip() if stem else line.strip()

    if current is not None:
        current["text"] = clean_text(current["text"])
        options.append(current)

    return clean_text(stem), options


# =========================================================
# 6. 单个 block 解析
# =========================================================
def remove_answer_span(text, span):
    if span is None:
        return clean_text(text)
    start, end = span
    new_text = (text[:start] + text[end:]).strip()
    return clean_text(new_text)


def extract_answer_from_solution_tail(text):
    text = clean_text(text)
    if text is None:
        return None

    boxed = list(BOXED_RE.finditer(text))
    if boxed:
        return normalize_answer(boxed[-1].group(1))

    return None


def parse_single_block(block, dataset_group):
    block = clean_text(block)
    if block is None:
        return {
            "question": None,
            "options": [],
            "answer": None,
        }

    if dataset_group == "ifeval":
        q, opts = split_question_and_options(block)
        return {
            "question": q,
            "options": opts,
            "answer": None,
        }

    answer, span = extract_explicit_answer(block)
    q_part = remove_answer_span(block, span)

    # 对数学/自由生成任务，若没有显式 answer，再尝试 boxed
    if answer is None and dataset_group in {"math_lvl5", "gsm8k", "bbh", "musr"}:
        answer = extract_answer_from_solution_tail(block)

    q_stem, options = split_question_and_options(q_part)

    return {
        "question": q_stem,
        "options": options,
        "answer": answer,
    }


# =========================================================
# 7. prompt 解析主函数
# =========================================================
def parse_prompt(prompt, dataset_group):
    blocks = split_prompt_into_blocks(prompt, dataset_group)

    parsed = [parse_single_block(block, dataset_group) for block in blocks]

    # 清理空块
    cleaned = []
    for item in parsed:
        q = clean_text(item["question"])
        opts = item["options"] if item["options"] else []
        a = item["answer"]

        if q is None and not opts and a is None:
            continue

        cleaned.append({
            "question": q,
            "options": opts,
            "answer": a,
        })

    # IFEval 固定只保留 1 个整体 prompt
    if dataset_group == "ifeval":
        if not cleaned:
            return [None], [[]], [None]
        item = cleaned[0]
        return [item["question"]], [item["options"]], [None]

    questions = [x["question"] for x in cleaned]
    options_list = [x["options"] for x in cleaned]
    answers = [x["answer"] for x in cleaned]

    if not questions:
        return [clean_text(prompt)], [[]], [None]

    return questions, options_list, answers


# =========================================================
# 8. 生成 DataFrame
# =========================================================
def build_prompt_qa_table(prompt_records):
    rows = []

    for i, rec in enumerate(prompt_records):
        prompt = rec["prompt"]
        dataset = rec["dataset_group"]
        dataset_name = rec["dataset_name"]

        questions, options_list, answers = parse_prompt(prompt, dataset)

        rows.append({
            "item_id": i,
            "dataset": dataset,
            "dataset_name": dataset_name,
            "question": json_dumps(questions),
            "options": json_dumps(options_list),
            "answer": json_dumps(answers),
            "num_qa_pairs": len(questions),
            "has_options": any(len(x) > 0 for x in options_list),
            "has_missing_answer": any(a is None for a in answers),
            "raw_prompt": clean_text(prompt),
        })

    return pd.DataFrame(rows)


def save_prompt_qa_csv(prompt_records, output_csv_path):
    df = build_prompt_qa_table(prompt_records)
    #df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    return df


# =========================================================
# 9. 一步跑完
# =========================================================
def export_prompt_question_option_answer_csv(
    prompt_old,
    prompt_new,
    prompt_mmlu_pro,
    output_csv_path="prompt_question_options_answer_dataset.csv",
):
    prompt_records = build_prompt_records(prompt_old, prompt_new, prompt_mmlu_pro)
    df = save_prompt_qa_csv(prompt_records, output_csv_path)
    return df


# =========================================================
# 10. 抽检工具
# =========================================================
def inspect_rows(df, dataset=None, n=3):
    sub = df if dataset is None else df[df["dataset"] == dataset]
    sub = sub.head(n)

    for _, row in sub.iterrows():
        print("=" * 100)
        print("item_id:", row["item_id"])
        print("dataset:", row["dataset"])
        print("dataset_name:", row["dataset_name"])
        print("num_qa_pairs:", row["num_qa_pairs"])
        print("has_options:", row["has_options"])
        print("has_missing_answer:", row["has_missing_answer"])

        questions = json.loads(row["question"])
        options = json.loads(row["options"])
        answers = json.loads(row["answer"])

        for j, (q, opt, a) in enumerate(zip(questions, options, answers)):
            print("-" * 80)
            print(f"QA #{j}")
            print("QUESTION:")
            print(q)
            print("OPTIONS:")
            print(opt)
            print("ANSWER:")
            print(a)


# =========================================================
# 11. 解析成功率检查
# =========================================================
def dataset_parse_summary(df):
    summary = (
        df.groupby("dataset")
        .agg(
            n_prompts=("item_id", "count"),
            avg_num_qa_pairs=("num_qa_pairs", "mean"),
            pct_has_options=("has_options", "mean"),
            pct_has_missing_answer=("has_missing_answer", "mean"),
        )
        .reset_index()
    )
    return summary


# =========================================================
# 12. 用法示例
# =========================================================
# df = export_prompt_question_option_answer_csv(
#     prompt_old=prompt_old,
#     prompt_new=prompt_new,
#     prompt_mmlu_pro=prompt_mmlu_pro,
#     output_csv_path="prompt_question_options_answer_dataset.csv",
# )
#
# print(df.head())
# print(dataset_parse_summary(df))
# inspect_rows(df, dataset="mmlu", n=2)

In [20]:
qa_df = export_prompt_question_option_answer_csv(
     prompt_old=prompt_old,
     prompt_new=prompt_new,
     prompt_mmlu_pro=prompt_mmlu_pro,
     output_csv_path="prompts_details.csv",
)

print(qa_df.head())
print(dataset_parse_summary(qa_df))
inspect_rows(qa_df, dataset="mmlu", n=2)

   item_id     dataset          dataset_name  \
0        0  winogrande  harness_winogrande_5   
1        1  winogrande  harness_winogrande_5   
2        2  winogrande  harness_winogrande_5   
3        3  winogrande  harness_winogrande_5   
4        4  winogrande  harness_winogrande_5   

                                            question options  answer  \
0  ["Natalie took basic French lessons from Betty...    [[]]  [null]   
1  ["Ryan has more stress levels in their body th...    [[]]  [null]   
2  ["Mary poured the entire bottle of cleaning so...    [[]]  [null]   
3  ["Aaron needed help from Samuel with their die...    [[]]  [null]   
4  ["To save the ship, we had to lose the anchor ...    [[]]  [null]   

   num_qa_pairs  has_options  has_missing_answer  \
0             1        False                True   
1             1        False                True   
2             1        False                True   
3             1        False                True   
4             1   

# 3.Load LLMs

In [2]:
with open(f'C:\\Users\\zqy\\OneDrive\\leaderboard_new.pkl', 'rb') as f:
    new_answer = pickle.load(f)
with open(f'C:\\Users\\zqy\\OneDrive\\leaderboard_old.pkl', 'rb') as f:
    old_answer = pickle.load(f)
with open(f'C:\\Users\\zqy\\OneDrive\\leaderboard_mmlu_pro.pkl', 'rb') as f:
    mmlu_pro_answer = pickle.load(f)

In [3]:
model_list = []
for i in old_answer['model']:
    model_list.append(i)
for i in new_answer['model']:
    model_list.append(i)
for i in mmlu_pro_answer['model']:
    model_list.append(i)
print(len(model_list))
print(model_list)

10634
['open-llm-leaderboard-old/details_Locutusque__Hyperion-1.5-Mistral-7B', 'open-llm-leaderboard-old/details_rhaymison__Mistral-portuguese-luana-7b-chat', 'open-llm-leaderboard-old/details_allknowingroger__Platapus-Orca-13B', 'open-llm-leaderboard-old/details_liuchanghf__phi2-mmlu-lora', 'open-llm-leaderboard-old/details_MaziyarPanahi__Calme-4x7B-MoE-v0.2', 'open-llm-leaderboard-old/details_bardsai__jaskier-7b-dpo-v7.1', 'open-llm-leaderboard-old/details_yam-peleg__Experiment24-7B', 'open-llm-leaderboard-old/details_allknowingroger__StarlingMaxLimmy2-7B-slerp', 'open-llm-leaderboard-old/details_robinsmits__Qwen1.5-7B-Dutch-Chat-Sft-Bf16', 'open-llm-leaderboard-old/details_DreadPoor__Sphinx-7B-Model_Stock', 'open-llm-leaderboard-old/details_Isaak-Carter__JOSIE_Beta-3-7B-slerp', 'open-llm-leaderboard-old/details_wandb__pruned_mistral', 'open-llm-leaderboard-old/details_frankenmerger__delta-4B-scientific', 'open-llm-leaderboard-old/details_automerger__ShadowYamshadow-7B', 'open-llm-le

# 4.Merge Matrix

In [4]:
import numpy as np
arr_list=[]
target_cols = 10634
sum = 0
for i in ['harness_winogrande_5', 'harness_hendrycksTest_high_school_biology_5', 'harness_hendrycksTest_philosophy_5', 'harness_hendrycksTest_professional_accounting_5', 'harness_hendrycksTest_human_aging_5', 'harness_hendrycksTest_clinical_knowledge_5', 'harness_hendrycksTest_college_medicine_5', 'harness_hendrycksTest_virology_5', 'harness_hendrycksTest_anatomy_5', 'harness_arc_challenge_25', 'harness_hendrycksTest_public_relations_5', 'harness_hendrycksTest_human_sexuality_5', 'harness_hendrycksTest_security_studies_5', 'harness_hendrycksTest_elementary_mathematics_5', 'harness_hendrycksTest_medical_genetics_5', 'harness_hendrycksTest_high_school_chemistry_5', 'harness_hendrycksTest_prehistory_5', 'harness_hendrycksTest_us_foreign_policy_5', 'harness_hendrycksTest_high_school_microeconomics_5', 'harness_hendrycksTest_high_school_government_and_politics_5', 'harness_hendrycksTest_computer_security_5', 'harness_hendrycksTest_global_facts_5', 'harness_gsm8k_5', 'harness_hendrycksTest_high_school_world_history_5', 'harness_hendrycksTest_jurisprudence_5', 'harness_hendrycksTest_astronomy_5', 'harness_hendrycksTest_college_computer_science_5', 'harness_hendrycksTest_high_school_computer_science_5', 'harness_hendrycksTest_college_mathematics_5', 'harness_hendrycksTest_sociology_5', 'harness_hendrycksTest_machine_learning_5', 'harness_hendrycksTest_conceptual_physics_5', 'harness_hendrycksTest_high_school_mathematics_5', 'harness_hendrycksTest_high_school_macroeconomics_5', 'harness_hendrycksTest_high_school_us_history_5', 'harness_hendrycksTest_professional_psychology_5', 'harness_hendrycksTest_moral_disputes_5', 'harness_hendrycksTest_management_5', 'harness_hendrycksTest_international_law_5', 'harness_hendrycksTest_high_school_physics_5', 'harness_truthfulqa_mc_0', 'harness_hendrycksTest_logical_fallacies_5', 'harness_hendrycksTest_world_religions_5', 'harness_hendrycksTest_professional_medicine_5', 'harness_hendrycksTest_college_physics_5', 'harness_hendrycksTest_formal_logic_5', 'harness_hendrycksTest_nutrition_5', 'harness_hendrycksTest_econometrics_5', 'harness_hendrycksTest_high_school_geography_5', 'harness_hendrycksTest_high_school_psychology_5', 'harness_hendrycksTest_electrical_engineering_5', 'harness_hellaswag_10', 'harness_hendrycksTest_high_school_statistics_5', 'harness_hendrycksTest_high_school_european_history_5', 'harness_hendrycksTest_marketing_5', 'harness_hendrycksTest_moral_scenarios_5', 'harness_hendrycksTest_business_ethics_5', 'harness_hendrycksTest_miscellaneous_5', 'harness_hendrycksTest_professional_law_5', 'harness_hendrycksTest_college_biology_5', 'harness_hendrycksTest_abstract_algebra_5', 'harness_hendrycksTest_college_chemistry_5']:

    mat =  old_answer['data'][i]['correctness']

    rows, cols = mat.shape
    
    if cols < target_cols:
        # 需要补的列数
        add_cols = target_cols - cols
        
        # 创建填充矩阵
        padding = np.full((rows, add_cols), -1)
        
        # 拼接
        old_answer['data'][i]['correctness'] = np.hstack((mat, padding))

    #print(old_answer['data'][i]['correctness'].shape)
    sum+=old_answer['data'][i]['correctness'].shape[0]
    arr_list.append(old_answer['data'][i]['correctness'])
print(sum)

for i in ['gpqa_extended', 'bbh_formal_fallacies', 'ifeval', 'bbh_tracking_shuffled_objects_five_objects', 'math_algebra_hard', 'musr_murder_mysteries', 'bbh_disambiguation_qa', 'bbh_hyperbaton', 'gpqa_main', 'bbh_geometric_shapes', 'bbh_object_counting', 'bbh_web_of_lies', 'bbh_tracking_shuffled_objects_three_objects', 'math_intermediate_algebra_hard', 'bbh_movie_recommendation', 'musr_team_allocation', 'bbh_logical_deduction_seven_objects', 'math_prealgebra_hard', 'bbh_boolean_expressions', 'gpqa_diamond', 'bbh_ruin_names', 'bbh_penguins_in_a_table', 'bbh_snarks', 'bbh_logical_deduction_five_objects', 'bbh_logical_deduction_three_objects', 'bbh_navigate', 'math_geometry_hard', 'bbh_reasoning_about_colored_objects', 'math_num_theory_hard', 'bbh_causal_judgement', 'musr_object_placements', 'bbh_tracking_shuffled_objects_seven_objects', 'math_precalculus_hard', 'bbh_sports_understanding', 'math_counting_and_prob_hard', 'bbh_date_understanding', 'bbh_temporal_sequences', 'bbh_salient_translation_error_detection']:


    mat = new_answer['data'][i]['correctness']

    rows, cols = mat.shape

    # 1. 左侧增加5000列 -1
    left_pad = np.full((rows, 5000), -1)

    # 2. 先拼接左侧
    arr = np.hstack((left_pad, mat))

    # 3. 右侧补到10634列（填-1）
    current_cols = arr.shape[1]
    #print(arr.shape[1])
    if current_cols < target_cols:
        right_pad = np.full((rows, target_cols - current_cols), -1)
        arr = np.hstack((arr, right_pad))

    # 写回原结构
    new_answer['data'][i]['correctness'] = arr

    #print(new_answer['data'][i]['correctness'].shape)
    sum+=new_answer['data'][i]['correctness'].shape[0]
    arr_list.append(new_answer['data'][i]['correctness'])
print(sum)
    
mat =  mmlu_pro_answer['data']['mmlu_pro']['correctness']
rows, cols = mat.shape
if cols < target_cols:
    # 需要补的列数
    add_cols = target_cols - cols

    # 创建填充矩阵
    padding = np.full((rows, add_cols), -1)

    # 拼接
    mmlu_pro_answer['data']['mmlu_pro']['correctness'] = np.hstack((padding,mat))
    sum+=mmlu_pro_answer['data']['mmlu_pro']['correctness'].shape[0]
    arr_list.append(mmlu_pro_answer['data']['mmlu_pro']['correctness'])
#print(mmlu_pro_answer['data']['mmlu_pro']['correctness'].shape)
print(sum)

28659
38233
50265


In [5]:
score_arr = np.vstack(arr_list)
print(score_arr.shape)

(50265, 10634)


In [6]:
import numpy as np
llm_name_list = np.concatenate([
    old_answer['model'],
    new_answer['model'],
    mmlu_pro_answer['model']
])
print(llm_name_list)

['open-llm-leaderboard-old/details_Locutusque__Hyperion-1.5-Mistral-7B'
 'open-llm-leaderboard-old/details_rhaymison__Mistral-portuguese-luana-7b-chat'
 'open-llm-leaderboard-old/details_allknowingroger__Platapus-Orca-13B' ...
 'jaspionjader__kstc-5-8b-details'
 'T145__ZEUS-8B-V17-abliterated-V4-details'
 'JayHyeon__Qwen2.5-0.5B-SFT-2e-5-2ep-DPOP_3e-7-3ep_0alp_5lam-details']


In [7]:
import numpy as np
from collections import OrderedDict

def normalize_model_name(name):
    """Strip prefix/suffix differences between old and new leaderboard naming."""
    name = name.replace("open-llm-leaderboard-old/", "")
    if name.startswith("details_"):
        name = name[len("details_"):]
    if name.endswith("-details"):
        name = name[:-len("-details")]
    return name

def merge_duplicate_llm_columns(score_arr, llm_name_list):
    """
    合并 score_arr 中对应同一个 LLM 的重复列。
    
    参数
    ----
    score_arr : np.ndarray, shape = (n_questions, n_models)
        每列对应一个模型，每行对应一个题目，值通常为:
        -1 表示缺失
         0 表示答错
         1 表示答对
         
    llm_name_list : array-like, shape = (n_models,)
        与 score_arr 列一一对应的模型名列表
    
    返回
    ----
    merged_score_arr : np.ndarray
        合并重名 LLM 后的新矩阵
        
    merged_llm_name_list : np.ndarray
        合并后的模型名列表（标准化后的名字）
    """
    llm_name_list = np.asarray(llm_name_list)

    if score_arr.shape[1] != len(llm_name_list):
        raise ValueError(
            f"score_arr 的列数 ({score_arr.shape[1]}) 和 llm_name_list 长度 ({len(llm_name_list)}) 不一致"
        )

    # 1. 按标准化后的模型名收集列索引
    name_to_indices = OrderedDict()
    for idx, name in enumerate(llm_name_list):
        norm_name = normalize_model_name(name)
        if norm_name not in name_to_indices:
            name_to_indices[norm_name] = []
        name_to_indices[norm_name].append(idx)

    # 2. 逐组合同名列
    merged_cols = []
    merged_names = []

    for norm_name, indices in name_to_indices.items():
        submat = score_arr[:, indices]   # shape: (n_rows, n_dup_cols)

        if len(indices) == 1:
            # 只有一列，不需要合并
            merged_col = submat[:, 0].copy()
        else:
            # 先默认都是 -1
            merged_col = np.full(submat.shape[0], -1, dtype=score_arr.dtype)

            # 找出每行第一个非 -1 的位置
            valid_mask = (submat != -1)  # True 表示该位置有真实值 0/1
            has_valid = valid_mask.any(axis=1)

            # argmax 会返回每行第一个 True 的位置
            first_valid_idx = valid_mask.argmax(axis=1)

            row_idx = np.where(has_valid)[0]
            col_idx = first_valid_idx[has_valid]

            merged_col[has_valid] = submat[row_idx, col_idx]

            # 可选：检查同名列中是否有冲突（同一行既出现0又出现1）
            for r in row_idx:
                vals = submat[r, submat[r] != -1]
                if len(vals) > 1 and np.any(vals != vals[0]):
                    raise ValueError(
                        f"模型 '{norm_name}' 在第 {r} 行出现冲突值: {vals}"
                    )

        merged_cols.append(merged_col)
        merged_names.append(norm_name)

    # 3. 拼成新的矩阵
    merged_score_arr = np.column_stack(merged_cols)
    merged_llm_name_list = np.array(merged_names)

    return merged_score_arr, merged_llm_name_list

In [8]:
merged_score_arr, merged_llm_name_list = merge_duplicate_llm_columns(
    score_arr,
    llm_name_list
)

print("原始 score_arr shape:", score_arr.shape)
print("合并后 score_arr shape:", merged_score_arr.shape)
print("原始 llm 数量:", len(llm_name_list))
print("合并后 llm 数量:", len(merged_llm_name_list))

原始 score_arr shape: (50265, 10634)
合并后 score_arr shape: (50265, 8577)
原始 llm 数量: 10634
合并后 llm 数量: 8577


In [11]:
import json

# 先转成普通 Python list + 全部转字符串
llm_name_list = [str(x) for x in merged_llm_name_list.tolist()]

# 保存成 json
with open("llm_names.json", "w", encoding="utf-8") as f:
    json.dump(llm_name_list, f, ensure_ascii=False, indent=2)

print(f"Saved {len(llm_name_list)} names to llm_names.json")

Saved 8577 names to llm_names.json


# 5.LLMs Tagging

In [14]:
from __future__ import annotations
import re, math, os
from collections import Counter
from typing import List, Dict, Optional, Tuple
import pandas as pd

def normalize_name(name: str) -> str:
    s = str(name).strip().replace("\\", "/")
    s = re.sub(r"\s+", " ", s)
    return s

def lower_name(name: str) -> str:
    return normalize_name(name).lower()

def get_core_name(name: str) -> str:
    s = lower_name(name)
    if "__" in s:
        s = s.split("__", 1)[1]
    if "/" in s:
        s = s.split("/")[-1]
    return s

BASE_FAMILY_RULES = [
    ("qwen", [r"qwen(?:[-_. ]?\d+(?:\.\d+)?)?", r"\bqwq\b"]),
    ("llama", [r"tinyllama", r"llama(?:[-_. ]?\d(?:\.\d+)?)?"]),
    ("mistral", [r"mixtral", r"openmixtral", r"biomistral", r"mistralmath", r"mistral", r"codestral", r"mathstral", r"ministral"]),
    ("gemma", [r"codegemma", r"recurrentgemma", r"shieldgemma", r"pali[-_. ]?gemma", r"gemma"]),
    ("phi", [r"(?<![a-z])phi[-_. ]?[234](?:\.\d+)?", r"(?<![a-z])phi[234](?![a-z])", r"(?<![a-z])phi(?![a-z])"]),
    ("deepseek", [r"deepseek", r"\bjanus\b"]),
    ("yi", [r"(?<![a-z])yi[-_. ]?\d", r"(?<![a-z])yi(?![a-z])"]),
    ("chatglm", [r"chatglm", r"(?<![a-z])glm[-_. ]?4"]),
    ("internlm", [r"internlm"]),
    ("baichuan", [r"baichuan"]),
    ("falcon", [r"falcon"]),
    ("mpt", [r"(?<![a-z])mpt[-_. ]?\d", r"(?<![a-z])mpt(?![a-z])"]),
    ("olmo", [r"olmo"]),
    ("pythia", [r"pythia", r"gpt[-_. ]?neox"]),
    ("bloom", [r"bloom"]),
    ("opt", [r"(?<![a-z])opt[-_. ]?\d", r"facebook/opt"]),
    ("mamba", [r"(?<![a-z])mamba(?![a-z])", r"jamba"]),
    ("stablelm", [r"stablelm"]),
    ("rwkv", [r"rwkv"]),
    ("xverse", [r"xverse"]),
    ("aquila", [r"aquila"]),
    ("minicpm", [r"minicpm"]),
    ("nemotron", [r"nemotron"]),
    ("granite", [r"granite"]),
    ("exaone", [r"exaone"]),
    ("cohere-command", [r"command[-_. ]?r\+?", r"cohere", r"(?<![a-z])aya(?![a-z])"]),
    ("zephyr", [r"zephyr"]),
    ("openchat", [r"openchat"]),
    ("vicuna", [r"vicuna"]),
    ("alpaca", [r"alpaca"]),
    ("orca", [r"orca"]),
    ("nous-hermes", [r"nous[-_. ]?hermes", r"(?<![a-z])hermes(?![a-z])"]),
    ("wizardlm", [r"wizard[-_. ]?(?:lm|math|coder)?"]),
    ("starling", [r"starling"]),
    ("solar", [r"(?<![a-z])solar(?![a-z])"]),
    ("dolphin", [r"dolphin"]),
    ("smollm", [r"smollm\d*"]),
    ("kosmos", [r"kosmos"]),
    ("airoboros", [r"airoboros"]),
    ("redpajama", [r"redpajama"]),
    ("gpt2", [r"(?<![a-z])gpt[-_. ]?2(?![a-z])"]),
]

COMPILED_BASE_FAMILY_RULES = [(fam, [re.compile(p) for p in pats]) for fam, pats in BASE_FAMILY_RULES]

VARIANT_PATTERNS = {
    "instruct": [r"\binstruct\b", r"(?<![a-z])it(?![a-z])"],
    "chat": [r"\bchat\b"],
    "base": [r"\bbase\b", r"\bpretrain(?:ed)?\b"],
    "moe": [r"\bmoe\b", r"mixtral", r"\d+x\d+(?:\.\d+)?b\b"],
    "coder": [r"\bcoder\b", r"(?<![a-z])code(?![a-z])", r"\bprogram\b"],
    "math": [r"\bmath\b", r"wizardmath", r"mathstral"],
    "vision": [r"\bvision\b", r"(?<![a-z])vl(?![a-z])", r"\bmultimodal\b", r"\bkosmos\b"],
    "reasoning": [r"\breason(?:ing)?\b", r"(?<![a-z])r1(?![a-z])", r"\bthink(?:ing)?\b"],
    "multilingual": [r"\bmultilingual\b", r"(?<![a-z])aya(?![a-z])"],
    "sft": [r"\bsft\b"],
    "dpo": [r"\bdpo\b"],
    "rlhf": [r"\brlhf\b"],
    "quantized": [r"\bawq\b", r"\bgptq\b", r"\bgguf\b", r"\bint4\b", r"\bint8\b", r"\b4bit\b", r"\b8bit\b"],
}

DOMAIN_PATTERNS = {
    "medical": [r"\bmedical\b", r"\bmedicine\b", r"\bmeditron\b", r"\bbiomed(?:ical)?\b", r"\bclinical\b", r"\bhealth(?:care)?\b", r"\bdoctor\b"],
    "law": [r"\blaw\b", r"\blegal\b", r"\battorney\b", r"\bjuris\b"],
    "education": [r"\bedu(?:cation)?\b", r"\btutor\b", r"\bteaching\b", r"\blearn(?:er)?\b"],
    "finance": [r"\bfinance\b", r"\bfinancial\b", r"\bbank\b", r"\btrader\b", r"\bstock\b", r"\bquant\b"],
    "biology": [r"\bbiology\b", r"\bgenome\b", r"\bprotein\b"],
    "chemistry": [r"\bchem(?:istry)?\b", r"\bmolecule\b"],
    "code": [r"\bcoder\b", r"(?<![a-z])code(?![a-z])", r"\bprogram\b", r"\bsoftware\b"],
    "math": [r"\bmath\b", r"\bgaokao\b", r"\bolympiad\b", r"\barithmetic\b"],
    "vision": [r"\bvision\b", r"(?<![a-z])vl(?![a-z])", r"\bmultimodal\b", r"\bimage\b", r"\bkosmos\b"],
    "reasoning": [r"\breason(?:ing)?\b", r"(?<![a-z])r1(?![a-z])", r"\bthink(?:ing)?\b"],
}
APP_DOMAIN = {"medical", "law", "education", "finance", "biology", "chemistry"}
CAP_SPECIAL = {"code", "math", "vision", "reasoning"}

SERIES_STOPWORDS = set("""
chat instruct base it hf gguf awq gptq int4 int8 4bit 8bit preview mini small medium large xl xxl plus pro text generation model models llm lm ai turbo alpha beta
sft dpo orpo rlhf kto ipo simpo cpo spin merge merged slerp ties dare ece linear stock test iter lora moe quantized bf16 fp16 qlora full passthrough raw pruned
v0 v1 v2 v3 v4 v5 v6 latest main old new final
""".split())

def infer_base_family(name: str) -> str:
    text = get_core_name(name) + " || " + lower_name(name)
    for fam, pats in COMPILED_BASE_FAMILY_RULES:
        for pat in pats:
            if pat.search(text):
                return fam
    return "unknown"

def infer_variant_tags(name: str) -> List[str]:
    s = lower_name(name)
    out = []
    for tag, pats in VARIANT_PATTERNS.items():
        if any(re.search(p, s) for p in pats):
            out.append(tag)
    return sorted(set(out))

def infer_domain_tags(name: str) -> List[str]:
    s = lower_name(name)
    out = []
    for tag, pats in DOMAIN_PATTERNS.items():
        if any(re.search(p, s) for p in pats):
            out.append(tag)
    return sorted(set(out))

def is_domain_specific(domain_tags: List[str], strict_application_only: bool = False) -> bool:
    tag_set = set(domain_tags)
    if strict_application_only:
        return len(tag_set & APP_DOMAIN) > 0
    return len(tag_set & (APP_DOMAIN | CAP_SPECIAL)) > 0

def extract_size_b(name: str) -> Optional[float]:
    s = lower_name(name)
    for pat in [r'(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*b\b', r'(\d+(?:\.\d+)?)x(\d+(?:\.\d+)?)b\b']:
        m = re.search(pat, s)
        if m:
            return round(float(m.group(1)) * float(m.group(2)), 4)
    b_vals = [float(m.group(1)) for m in re.finditer(r'(?<![a-z])(\d+(?:\.\d+)?)\s*b\b', s)]
    if b_vals:
        return max(b_vals)
    m_vals = [float(m.group(1))/1000.0 for m in re.finditer(r'(?<![a-z])(\d+(?:\.\d+)?)\s*m\b', s)]
    if m_vals:
        return max(m_vals)
    return None

def bucket_size(size_b: Optional[float]) -> str:
    if size_b is None:
        return "unknown"
    if size_b < 3: return "<3B"
    if size_b < 7: return "3-7B"
    if size_b < 13: return "7-13B"
    if size_b < 34: return "13-34B"
    if size_b < 80: return "34-80B"
    return "80B+"

def infer_series_family(name: str) -> str:
    base = infer_base_family(name)
    if base != "unknown":
        return base
    s = get_core_name(name)
    toks = [t for t in re.split(r"[\/_\-\.\(\)\[\],: ]+", s) if t]
    cleaned = []
    for t in toks:
        if t in SERIES_STOPWORDS:
            continue
        if re.fullmatch(r'\d+(?:\.\d+)?[bm]|[lv]\d+|\d+k|\d+', t):
            continue
        if re.fullmatch(r'\d+x\d+(?:\.\d+)?b', t):
            continue
        cleaned.append(t)
    return cleaned[0] if cleaned else "unknown"

def analyze_models(llm_name_list: List[str], out_dir: Optional[str] = None, strict_application_only: bool = False) -> pd.DataFrame:
    cleaned = list(dict.fromkeys([normalize_name(x) for x in llm_name_list if str(x).strip()]))
    rows = []
    for name in cleaned:
        base_family = infer_base_family(name)
        series_family = infer_series_family(name)
        variant_tags = infer_variant_tags(name)
        domain_tags = infer_domain_tags(name)
        size_b = extract_size_b(name)
        rows.append({
            "original_name": name,
            "core_name": get_core_name(name),
            "base_family": base_family,
            "series_family": series_family,
            "variant_tags": ";".join(variant_tags),
            "domain_tags": ";".join(domain_tags),
            "size_b": size_b,
            "size_bucket": bucket_size(size_b),
            "is_domain_specific": is_domain_specific(domain_tags, strict_application_only),
            "is_general": not is_domain_specific(domain_tags, strict_application_only),
        })
    df = pd.DataFrame(rows)

    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(os.path.join(out_dir, "llm_metadata_improved.csv"), index=False, encoding="utf-8-sig")
        (
            df["base_family"].value_counts().rename_axis("base_family").reset_index(name="count")
        ).to_csv(os.path.join(out_dir, "base_family_counts_improved.csv"), index=False, encoding="utf-8-sig")
        (
            df["series_family"].value_counts().rename_axis("series_family").reset_index(name="count")
        ).to_csv(os.path.join(out_dir, "series_family_counts_improved.csv"), index=False, encoding="utf-8-sig")
    return df

In [15]:
df_llm = analyze_models(
        merged_llm_name_list,
        out_dir="llm_name_analysis",
        strict_application_only=False,
    )


In [ ]:
# 直接对已有 df_llm 做 Hugging Face enrich
# pip install pandas requests huggingface_hub tqdm

from __future__ import annotations

import re
import time
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from huggingface_hub import HfApi
from tqdm import tqdm


# =========================================================
# 1) 基础工具
# =========================================================

def normalize_name(name: str) -> str:
    s = str(name).strip()
    s = s.replace("\\", "/")
    s = re.sub(r"\s+", " ", s)
    return s


def simplify_for_match(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("_", "-")
    s = re.sub(r"[^a-z0-9\-./]+", "-", s)
    s = re.sub(r"-+", "-", s).strip("-")
    return s


def safe_getattr(obj: Any, attr: str, default=None):
    try:
        return getattr(obj, attr, default)
    except Exception:
        return default


# =========================================================
# 2) domain 规则
#    这里只做 enrich 后的附加标签，不动你原来的列
# =========================================================

DOMAIN_PATTERNS = {
    "medical": [
        r"\bmed\b", r"\bmedical\b", r"\bmedicine\b", r"\bbiomed\b",
        r"\bbiomedical\b", r"\bhealth\b", r"\bdoctor\b", r"\bclinic\b",
        r"\bclinical\b", r"\bhealthcare\b",
    ],
    "law": [
        r"\blaw\b", r"\blegal\b", r"\battorney\b", r"\bjuris\b",
        r"\bjudicial\b", r"\bcase law\b",
    ],
    "education": [
        r"\bedu\b", r"\beducation\b", r"\btutor\b", r"\blearn\b",
        r"\bteach\b", r"\bteaching\b", r"\binstruction\b",
    ],
    "finance": [
        r"\bfinance\b", r"\bfinancial\b", r"\bbank\b", r"\btrader\b",
        r"\bstock\b", r"\bquant\b", r"\binvest\b",
    ],
    "science": [
        r"\bscience\b", r"\bscientific\b", r"\bchem\b",
        r"\bphysics\b", r"\bbio\b", r"\bbiology\b",
    ],
    "code": [
        r"\bcoder\b", r"\bcode\b", r"\bprogram\b", r"\bsoftware\b",
        r"\bdeveloper\b", r"\bdev\b",
    ],
    "math": [
        r"\bmath\b", r"\bmathematics\b", r"\barithmetic\b",
        r"\balgebra\b", r"\bgeometry\b", r"\bolympiad\b",
    ],
    "vision": [
        r"\bvision\b", r"\bvl\b", r"\bmultimodal\b", r"\bimage\b",
    ],
    "reasoning": [
        r"\breason\b", r"\breasoning\b", r"\br1\b", r"\bthinking\b",
    ],
}


def has_any_pattern(text: str, patterns: List[str]) -> bool:
    text = str(text).lower()
    return any(re.search(p, text) for p in patterns)


def infer_domain_tags(text: str) -> List[str]:
    s = str(text).lower()
    tags = []
    for tag, patterns in DOMAIN_PATTERNS.items():
        if has_any_pattern(s, patterns):
            tags.append(tag)
    return sorted(set(tags))


# =========================================================
# 3) Hugging Face repo 匹配
#    优先利用 df_llm 里已有 family 作为 hint
# =========================================================

def maybe_repo_id(name: str) -> Optional[str]:
    """
    如果字符串本身像 org/model，先当作 repo_id 试试。
    """
    name = normalize_name(name)
    if "/" in name and len(name.split("/")) == 2:
        return name
    return None


def choose_best_hf_match(
    model_name: str,
    candidates: List[Any],
    family_hint: Optional[str] = None,
) -> Optional[str]:
    if not candidates:
        return None

    raw = simplify_for_match(model_name)
    raw_base = raw.split("/")[-1]

    scored: List[Tuple[int, str]] = []

    for c in candidates:
        repo_id = c.id
        rid = simplify_for_match(repo_id)
        rid_base = rid.split("/")[-1]

        score = 0

        # 1) 精确匹配最优
        if rid == raw:
            score += 120
        if rid_base == raw_base:
            score += 60

        # 2) 包含关系
        if raw_base in rid_base:
            score += 25
        if rid_base in raw_base:
            score += 12

        # 3) family hint 加分
        if family_hint and family_hint != "unknown":
            fam = str(family_hint).lower().strip()
            if fam and fam in rid:
                score += 10

        # 4) repo 越短通常越像“主仓库”
        score -= max(0, len(rid_base) - len(raw_base)) * 0.05

        scored.append((score, repo_id))

    scored.sort(key=lambda x: x[0], reverse=True)
    best_score, best_repo = scored[0]

    # 分数太低时宁可不匹配
    return best_repo if best_score > 0 else None


def resolve_hf_repo(
    api: HfApi,
    model_name: str,
    family_hint: Optional[str] = None,
    sleep_sec: float = 0.05,
) -> Optional[str]:
    name = normalize_name(model_name)

    # 先试直接 repo_id
    direct = maybe_repo_id(name)
    if direct is not None:
        try:
            api.model_info(direct)
            return direct
        except Exception:
            pass

    # 再搜索 basename
    query = name.split("/")[-1] if "/" in name else name
    query = query.strip()
    if not query:
        return None

    try:
        candidates = list(api.list_models(search=query, limit=10))
        time.sleep(sleep_sec)
        return choose_best_hf_match(name, candidates, family_hint=family_hint)
    except Exception:
        return None


# =========================================================
# 4) 抓取 HF metadata
# =========================================================

def fetch_readme_raw(repo_id: str, timeout: int = 20) -> str:
    url = f"https://huggingface.co/{repo_id}/raw/main/README.md"
    r = requests.get(url, timeout=timeout)
    if r.status_code == 200:
        return r.text
    return ""


def extract_first_good_paragraph(readme_text: str) -> str:
    if not readme_text:
        return ""

    text = readme_text.strip()

    # 去掉 YAML front matter
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) == 3:
            text = parts[2].strip()

    paragraphs = re.split(r"\n\s*\n", text)

    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if p.startswith("#"):
            continue
        if p.startswith("![") or p.startswith("<img"):
            continue
        if p.startswith("|"):
            continue

        p = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", p)
        p = re.sub(r"`([^`]+)`", r"\1", p)
        p = re.sub(r"\s+", " ", p).strip()

        if len(p) >= 40:
            return p

    return ""


def fetch_hf_metadata(api: HfApi, repo_id: str) -> Dict[str, Any]:
    out = {
        "hf_repo_id": repo_id,
        "hf_description": "",
        "hf_tags": [],
        "hf_license": "",
        "hf_pipeline_tag": "",
        "hf_downloads": None,
        "hf_likes": None,
        "hf_library_name": "",
        "hf_model_index_present": False,
    }

    try:
        info = api.model_info(repo_id)

        out["hf_downloads"] = safe_getattr(info, "downloads", None)
        out["hf_likes"] = safe_getattr(info, "likes", None)
        out["hf_pipeline_tag"] = safe_getattr(info, "pipeline_tag", "") or ""
        out["hf_library_name"] = safe_getattr(info, "library_name", "") or ""

        tags = safe_getattr(info, "tags", None)
        if tags:
            out["hf_tags"] = list(tags)

        card_data = safe_getattr(info, "card_data", None)
        if card_data:
            try:
                card_data = dict(card_data) if not isinstance(card_data, dict) else card_data
                out["hf_license"] = str(card_data.get("license", "") or "")
                out["hf_model_index_present"] = "model-index" in card_data
            except Exception:
                pass

    except Exception:
        pass

    try:
        readme = fetch_readme_raw(repo_id)
        out["hf_description"] = extract_first_good_paragraph(readme)
    except Exception:
        pass

    return out


# =========================================================
# 5) enrich 单个模型
# =========================================================

def enrich_one_model_from_hf(
    model_name: str,
    family_hint: Optional[str] = None,
    api: Optional[HfApi] = None,
) -> Dict[str, Any]:
    if api is None:
        api = HfApi()

    model_name = normalize_name(model_name)

    row = {
        "original_name": model_name,
        "hf_repo_id": "",
        "hf_matched": False,
        "hf_description": "",
        "hf_tags": [],
        "hf_license": "",
        "hf_pipeline_tag": "",
        "hf_downloads": None,
        "hf_likes": None,
        "hf_library_name": "",
        "hf_model_index_present": False,
        "hf_domain_tags": [],
    }

    repo_id = resolve_hf_repo(api, model_name, family_hint=family_hint)
    if repo_id:
        meta = fetch_hf_metadata(api, repo_id)
        row.update(meta)
        row["hf_matched"] = True

        text_for_domain = " ".join([
            model_name,
            row["hf_description"],
            " ".join(row["hf_tags"]),
            row["hf_license"],
            row["hf_pipeline_tag"],
        ])
        row["hf_domain_tags"] = infer_domain_tags(text_for_domain)
    else:
        row["hf_domain_tags"] = infer_domain_tags(model_name)

    return row


# =========================================================
# 6) 主函数：直接对 df_llm enrich
# =========================================================

def enrich_df_llm_from_hf(
    df_llm: pd.DataFrame,
    name_col: str = "original_name",
    family_col: Optional[str] = "family",
    save_path: Optional[str] = None,
    cache_path: Optional[str] = None,
    sleep_sec_between_rows: float = 0.0,
) -> pd.DataFrame:
    """
    直接对已有 df_llm 做 HF enrich。

    参数
    ----
    df_llm : pd.DataFrame
        你已经跑好的 analyze_models 输出表
    name_col : str
        模型名列名，默认 original_name
    family_col : Optional[str]
        已有 family 列，若存在则用于 repo 匹配加分
    save_path : Optional[str]
        最终 enriched DataFrame 保存路径
    cache_path : Optional[str]
        单独保存 unique model -> HF enrich 结果，便于下次复用
    sleep_sec_between_rows : float
        每个 unique model enrich 后休眠秒数，防止过快

    返回
    ----
    df_out : pd.DataFrame
        在原 df_llm 基础上 merge 了 HF enrich 信息的新表
    """
    if name_col not in df_llm.columns:
        raise ValueError(f"name_col={name_col!r} 不在 df_llm 列中，现有列：{list(df_llm.columns)}")

    df = df_llm.copy()
    df[name_col] = df[name_col].astype(str).map(normalize_name)

    # 只对 unique model enrich 一次
    keep_cols = [name_col]
    if family_col is not None and family_col in df.columns:
        keep_cols.append(family_col)

    unique_df = df[keep_cols].drop_duplicates().reset_index(drop=True)

    # 如果有 cache，先读
    cache_df = None
    cached_names = set()
    if cache_path is not None:
        try:
            cache_df = pd.read_csv(cache_path)
            if name_col in cache_df.columns:
                cached_names = set(cache_df[name_col].astype(str).map(normalize_name))
        except Exception:
            cache_df = None
            cached_names = set()

    api = HfApi()
    rows = []

    # 先把 cache 里已有的拿出来
    if cache_df is not None and len(cached_names) > 0:
        cached_part = cache_df[cache_df[name_col].astype(str).map(normalize_name).isin(
            set(unique_df[name_col].tolist())
        )].copy()
        rows.extend(cached_part.to_dict("records"))

    # 找还没 enrich 的
    todo_df = unique_df[~unique_df[name_col].isin(cached_names)].reset_index(drop=True)

    for _, r in tqdm(todo_df.iterrows(), total=len(todo_df), desc="Enriching df_llm from Hugging Face"):
        model_name = r[name_col]
        family_hint = r[family_col] if (family_col is not None and family_col in r.index) else None

        out = enrich_one_model_from_hf(
            model_name=model_name,
            family_hint=family_hint,
            api=api,
        )
        rows.append(out)

        if sleep_sec_between_rows > 0:
            time.sleep(sleep_sec_between_rows)

    enrich_df = pd.DataFrame(rows)

    # list -> string，便于保存/merge
    for col in ["hf_tags", "hf_domain_tags"]:
        if col in enrich_df.columns:
            enrich_df[col] = enrich_df[col].apply(
                lambda x: ";".join(x) if isinstance(x, list) else ("" if pd.isna(x) else str(x))
            )

    # 统一列名以便 merge
    if "original_name" in enrich_df.columns and name_col != "original_name":
        enrich_df = enrich_df.rename(columns={"original_name": name_col})

    # 去重，防止 cache + 新跑重复
    enrich_df = enrich_df.drop_duplicates(subset=[name_col], keep="first").reset_index(drop=True)

    # 可选保存 cache
    if cache_path is not None:
        enrich_df.to_csv(cache_path, index=False, encoding="utf-8-sig")

    # merge 回原 df_llm
    df_out = df.merge(enrich_df, on=name_col, how="left")

    # 再补几个分析友好列
    if "hf_domain_tags" in df_out.columns:
        df_out["hf_is_domain_specific"] = df_out["hf_domain_tags"].fillna("").str.strip().ne("")
    else:
        df_out["hf_is_domain_specific"] = False

    if "hf_repo_id" in df_out.columns:
        df_out["hf_has_repo_match"] = df_out["hf_repo_id"].fillna("").str.strip().ne("")
    else:
        df_out["hf_has_repo_match"] = False

    if save_path is not None:
        df_out.to_csv(save_path, index=False, encoding="utf-8-sig")

    return df_out

In [ ]:
df_llm_enriched = enrich_df_llm_from_hf(
    df_llm,
    name_col="original_name",   
    save_path="df_llm_enriched.csv",
    cache_path="hf_enrich_cache.csv"
)

# 6.Items Filtering

In [16]:
import numpy as np
import pandas as pd

def filter_low_std_items(
    merged_score_arr,
    std_thresh=0.01,
    **aligned_objects
):
    """
    按 item 过滤标准差过低的题，并同步过滤所有与 item 对齐的信息。

    参数
    ----
    merged_score_arr : np.ndarray
        shape = (num_items, num_llms)
        值为 {-1, 0, 1}

    std_thresh : float
        标准差阈值，默认 0.01

    **aligned_objects :
        所有和 item 一一对应的对象，长度/行数必须等于 num_items。
        支持类型：
        - list
        - tuple
        - np.ndarray
        - pd.Series
        - pd.DataFrame

        例如：
        prompt_list=prompt_list,
        prompt_dataset_list=prompt_dataset_list,
        prompt_dataset_name_list=prompt_dataset_name_list,
        prompt_records=prompt_records,
        qa_df=qa_df

    返回
    ----
    filtered_arr : np.ndarray
        过滤后的 score matrix

    filtered_objects : dict
        所有同步过滤后的对象，key 与传入名字一致

    keep_mask : np.ndarray
        shape = (num_items,), True 表示保留

    item_std : np.ndarray
        每个 item 的标准差（无有效值的 item 为 np.nan）
    """
    arr = merged_score_arr
    assert isinstance(arr, np.ndarray), "merged_score_arr 必须是 np.ndarray"
    assert arr.ndim == 2, "merged_score_arr 必须是二维矩阵"

    num_items = arr.shape[0]

    # -------- 检查所有附带对象长度是否一致 --------
    for name, obj in aligned_objects.items():
        if isinstance(obj, (list, tuple, np.ndarray, pd.Series)):
            assert len(obj) == num_items, f"{name} 长度与 item 数量不一致: {len(obj)} != {num_items}"
        elif isinstance(obj, pd.DataFrame):
            assert len(obj) == num_items, f"{name} 行数与 item 数量不一致: {len(obj)} != {num_items}"
        else:
            raise TypeError(
                f"{name} 的类型 {type(obj)} 暂不支持。"
                "仅支持 list / tuple / np.ndarray / pd.Series / pd.DataFrame"
            )

    # -------- 逐行计算标准差：忽略 -1，只对 0/1 计算 --------
    item_std = np.full(num_items, np.nan, dtype=float)

    for i in range(num_items):
        row = arr[i]
        valid = row[row != -1]

        if len(valid) == 0:
            item_std[i] = np.nan
        else:
            item_std[i] = np.std(valid)

    # 保留标准差 >= 阈值，且不是 nan 的题
    keep_mask = (~np.isnan(item_std)) & (item_std >= std_thresh)
    removed_mask = ~keep_mask

    # -------- 过滤 score matrix --------
    filtered_arr = arr[keep_mask]

    # -------- 同步过滤所有关联对象 --------
    filtered_objects = {}

    for name, obj in aligned_objects.items():
        if isinstance(obj, list):
            filtered_objects[name] = [x for x, keep in zip(obj, keep_mask) if keep]

        elif isinstance(obj, tuple):
            filtered_objects[name] = tuple(x for x, keep in zip(obj, keep_mask) if keep)

        elif isinstance(obj, np.ndarray):
            filtered_objects[name] = obj[keep_mask]

        elif isinstance(obj, pd.Series):
            filtered_objects[name] = obj.loc[keep_mask].reset_index(drop=True)

        elif isinstance(obj, pd.DataFrame):
            filtered_objects[name] = obj.loc[keep_mask].reset_index(drop=True)

    print(f"Total items: {num_items}")
    print(f"Kept items: {keep_mask.sum()}")
    print(f"Removed items: {removed_mask.sum()}")
    print(f"Removed ratio: {removed_mask.sum() / num_items:.4f}")
    print(f"Removed by std < {std_thresh}: {np.sum((~np.isnan(item_std)) & (item_std < std_thresh))}")
    print(f"Removed by no valid responses: {np.sum(np.isnan(item_std))}")

    return filtered_arr, filtered_objects, keep_mask, item_std

In [21]:
filtered_arr, filtered_objects, keep_mask, item_std = filter_low_std_items(
    merged_score_arr,
    std_thresh=0.01,
    prompt_records=prompt_records,
    qa_df=qa_df
)

filtered_prompt_records = filtered_objects["prompt_records"]
filtered_qa_df = filtered_objects["qa_df"]

Total items: 50265
Kept items: 49828
Removed items: 437
Removed ratio: 0.0087
Removed by std < 0.01: 437
Removed by no valid responses: 0


In [22]:
merged_score_arr = filtered_arr
prompt_list = [x["prompt"] for x in filtered_prompt_records]

In [23]:
import numpy as np
import pandas as pd

def filter_extreme_items(
    merged_score_arr,
    high_thresh=0.98,
    low_thresh=0.02,
    **aligned_objects
):
    """
    按 item 过滤极端题，并同步过滤所有与 item 对齐的信息。

    参数
    ----
    merged_score_arr : np.ndarray
        shape = (num_items, num_llms)
        值为 {-1, 0, 1}

    high_thresh : float
        正确率高阈值，默认 0.98
        正确率 >= high_thresh 的题会被过滤

    low_thresh : float
        正确率低阈值，默认 0.02
        正确率 <= low_thresh 的题会被过滤

    **aligned_objects :
        所有和 item 一一对应的对象，长度/行数必须等于 num_items。
        支持类型：
        - list
        - tuple
        - np.ndarray
        - pd.Series
        - pd.DataFrame

        例如：
        prompt_list=prompt_list,
        prompt_dataset_list=prompt_dataset_list,
        prompt_dataset_name_list=prompt_dataset_name_list,
        prompt_records=prompt_records,
        qa_df=qa_df

    返回
    ----
    filtered_arr : np.ndarray
        过滤后的 score matrix

    filtered_objects : dict
        所有同步过滤后的对象，key 与传入名字一致

    keep_mask : np.ndarray
        shape = (num_items,), True 表示保留

    correct_ratio : np.ndarray
        每个 item 的正确率（无有效值的 item 为 np.nan）
    """
    arr = merged_score_arr
    assert isinstance(arr, np.ndarray), "merged_score_arr 必须是 np.ndarray"
    assert arr.ndim == 2, "merged_score_arr 必须是二维矩阵"

    num_items = arr.shape[0]

    # -------- 检查所有附带对象长度是否一致 --------
    for name, obj in aligned_objects.items():
        if isinstance(obj, (list, tuple, np.ndarray, pd.Series)):
            assert len(obj) == num_items, f"{name} 长度与 item 数量不一致: {len(obj)} != {num_items}"
        elif isinstance(obj, pd.DataFrame):
            assert len(obj) == num_items, f"{name} 行数与 item 数量不一致: {len(obj)} != {num_items}"
        else:
            raise TypeError(
                f"{name} 的类型 {type(obj)} 暂不支持。"
                "仅支持 list / tuple / np.ndarray / pd.Series / pd.DataFrame"
            )

    # -------- 1. 计算每个 item 的正确率 --------
    valid_mask = (arr != -1)
    valid_counts = valid_mask.sum(axis=1)
    correct_counts = ((arr == 1) & valid_mask).sum(axis=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        correct_ratio = correct_counts / valid_counts

    # 无有效作答的题标为 nan，便于统计
    correct_ratio = correct_ratio.astype(float)
    correct_ratio[valid_counts == 0] = np.nan

    # -------- 2. 构建过滤 mask --------
    # 保留“不是极端”的 items
    # 同时去掉没有有效作答的题
    keep_mask = (
        (valid_counts > 0) &
        (correct_ratio < high_thresh) &
        (correct_ratio > low_thresh)
    )
    removed_mask = ~keep_mask

    # -------- 3. 过滤 matrix --------
    filtered_arr = arr[keep_mask]

    # -------- 4. 同步过滤所有关联对象 --------
    filtered_objects = {}

    for name, obj in aligned_objects.items():
        if isinstance(obj, list):
            filtered_objects[name] = [x for x, keep in zip(obj, keep_mask) if keep]

        elif isinstance(obj, tuple):
            filtered_objects[name] = tuple(x for x, keep in zip(obj, keep_mask) if keep)

        elif isinstance(obj, np.ndarray):
            filtered_objects[name] = obj[keep_mask]

        elif isinstance(obj, pd.Series):
            filtered_objects[name] = obj.loc[keep_mask].reset_index(drop=True)

        elif isinstance(obj, pd.DataFrame):
            filtered_objects[name] = obj.loc[keep_mask].reset_index(drop=True)

    # -------- 5. 打印统计 --------
    total_items = num_items
    kept_items = keep_mask.sum()
    removed_items = removed_mask.sum()

    print(f"Total items: {total_items}")
    print(f"Kept items: {kept_items}")
    print(f"Removed items: {removed_items}")
    print(f"Removed ratio: {removed_items / total_items:.4f}")

    print(f"\nRemoved high (>= {high_thresh}): {np.sum((~np.isnan(correct_ratio)) & (correct_ratio >= high_thresh))}")
    print(f"Removed low  (<= {low_thresh}): {np.sum((~np.isnan(correct_ratio)) & (correct_ratio <= low_thresh))}")
    print(f"Removed by no valid responses: {np.sum(np.isnan(correct_ratio))}")

    return filtered_arr, filtered_objects, keep_mask, correct_ratio

In [25]:
filtered_arr, filtered_objects, keep_mask, correct_ratio = filter_extreme_items(
    merged_score_arr,
    high_thresh=0.98,
    low_thresh=0.02,
    prompt_records=filtered_prompt_records,
    qa_df=filtered_qa_df
)

Total items: 49828
Kept items: 44336
Removed items: 5492
Removed ratio: 0.1102

Removed high (>= 0.98): 1670
Removed low  (<= 0.02): 3822
Removed by no valid responses: 0


# 7.Generate prompts_details

In [26]:
filtered_prompt_records = filtered_objects["prompt_records"]
filtered_qa_df = filtered_objects["qa_df"].copy()
filtered_qa_df = filtered_qa_df.reset_index(drop=True)
filtered_qa_df["item_id"] = filtered_qa_df.index

In [27]:
filtered_qa_df.to_csv("C:\\Users\\zqy\\OneDrive\\桌面\\projet\\NeuralCDM\\data\\cdm-llm-evaluation-main\\cdm_exploration\\notebooks\\prompts_details.csv", index=False, encoding="utf-8-sig")

# 8.Generate response_matrix

In [28]:
import pandas as pd
import csv
response_df = pd.DataFrame(
    filtered_arr.T,  # (8577,44336)
    index=merged_llm_name_list,
    columns=[f"item_{i}" for i in range(filtered_arr.shape[0])],
)
response_df.index.name = "llm"

In [ ]:
response_df.to_csv("response_matrix.csv")
pd.Series(merged_llm_name_list, name="llm").to_csv("llm_list.csv", index=False)